In [1]:
# ============================================================
# Step 4 setup
#
# Upload:
#   1. config.py
#
# Read from Google Drive:
#   2. Step 3 zip, e.g. pilot3_step3A_diverse_text_preserve_text_direction.zip
#
# This cell only prepares files and directories.
# Detailed config and file-discovery checks are printed later.
# ============================================================

import os
import sys
import shutil
import zipfile
import importlib
from pathlib import Path
from google.colab import files
from google.colab import drive

drive.mount("/content/drive")

# ------------------------------------------------------------
# 1. Create project structure
# ------------------------------------------------------------

PILOT_ROOT = "/content/pilot_3"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")
DATA_DIR = os.path.join(PILOT_ROOT, "data")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Make pilot_3 importable as a package.
with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# ------------------------------------------------------------
# 2. Upload config.py
# ------------------------------------------------------------

print("Upload config.py")
uploaded_config = files.upload()

config_upload_name = None

for name in uploaded_config.keys():
    if name.startswith("config") and name.endswith(".py"):
        config_upload_name = name
        break

if config_upload_name is None:
    raise FileNotFoundError(
        f"No config*.py file was uploaded. Uploaded files: {list(uploaded_config.keys())}"
    )

config_target_path = os.path.join(PILOT_ROOT, "config.py")

shutil.copy(
    os.path.join("/content", config_upload_name),
    config_target_path,
)

# ------------------------------------------------------------
# 3. Load config and prepare directories
# ------------------------------------------------------------

import pilot_3.config as cfg
importlib.reload(cfg)

os.makedirs(cfg.STEP4_INPUT_DIR, exist_ok=True)

# Clear old Step 4 input files to avoid mixing stale Step 3 files.
for old_json in Path(cfg.STEP4_INPUT_DIR).glob("*_experiment_A_text_*.json"):
    old_json.unlink()

# Clear old Step 4 outputs to avoid stale files in the final zip.
if os.path.exists(cfg.STEP4_OUTPUT_DIR):
    shutil.rmtree(cfg.STEP4_OUTPUT_DIR)

os.makedirs(cfg.STEP4_OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 4. Use Step 3 zip from Google Drive
# ------------------------------------------------------------

step3_zip_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expA_settingA_6fps/settingA/data_expA_6fps_settingA/p4_settingA_step3/p4_settingA_step3.zip"
)

print("\nUsing Step 3 zip from Google Drive:")
print(step3_zip_path)

if not step3_zip_path.exists():
    raise FileNotFoundError(f"Step 3 zip not found: {step3_zip_path}")

zip_path = str(step3_zip_path)

# ------------------------------------------------------------
# 5. Unzip Step 3 output
# ------------------------------------------------------------

tmp_extract_dir = "/content/_step3_extract_tmp"

if os.path.exists(tmp_extract_dir):
    shutil.rmtree(tmp_extract_dir)

os.makedirs(tmp_extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(tmp_extract_dir)

# Copy Step 3 scene json files into cfg.STEP4_INPUT_DIR.
json_files = sorted(Path(tmp_extract_dir).rglob("*_experiment_A_text_*.json"))

if not json_files:
    raise FileNotFoundError(
        "No *_experiment_A_text_*.json files found in the Step 3 zip."
    )

copied_json_files = []

for jf in json_files:
    target_path = os.path.join(cfg.STEP4_INPUT_DIR, jf.name)
    shutil.copy(str(jf), target_path)
    copied_json_files.append(jf.name)

# ------------------------------------------------------------
# 6. Minimal setup summary
# ------------------------------------------------------------

print("\nSetup complete.")
print("Uploaded config:", config_upload_name)
print("Config copied to:", config_target_path)
print("Step 3 zip from Drive:", step3_zip_path)
print("Step 3 json files copied to:", cfg.STEP4_INPUT_DIR)
print("Number of Step 3 json files copied:", len(copied_json_files))
print("Step 4 output dir:", cfg.STEP4_OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Upload config.py


Saving config_exp_A_positional_context_6floorplans.py to config_exp_A_positional_context_6floorplans.py

Using Step 3 zip from Google Drive:
/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expA_settingA_6fps/settingA/data_expA_6fps_settingA/p4_settingA_step3/p4_settingA_step3.zip

Setup complete.
Uploaded config: config_exp_A_positional_context_6floorplans.py
Config copied to: /content/pilot_3/config.py
Step 3 zip from Drive: /content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expA_settingA_6fps/settingA/data_expA_6fps_settingA/p4_settingA_step3/p4_settingA_step3.zip
Step 3 json files copied to: /content/pilot_3/data/step3A_diverse_text_preserve_text_direction
Number of Step 3 json files copied: 6
Step 4 output dir: /content/pilot_3/data/step4_probe_datasets_preserve_text_direction


In [2]:
# ============================================================
# Imports and Step 4 configuration
#
# All configurable parameters are loaded from pilot_3.config.
# Do not redefine Step 4 parameters in this notebook.
#
# This cell prints only a concise config summary.
# Step 3 file discovery is printed later when files are processed.
# ============================================================

import os
import sys
import json
import random
import hashlib
import shutil
import importlib
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_3.config as cfg
importlib.reload(cfg)

from pilot_3.config import (
    RANDOM_SEED,
    SCENES,
    RUN_MODE,

    STEP4_INPUT_DIR,
    STEP4_OUTPUT_DIR,

    NONE_RELATION_LABEL,
    INVERSE_RELATION_MAP,
    DIRECTIONAL_RELATIONS,
    SYMMETRIC_OR_WEAK_RELATIONS,

    EXPLICIT_DIRECT,
    EXPLICIT_INVERSE,
    IMPLICIT_LABELED,
    IMPLICIT_NONE,

    MAIN_PROBE_EVIDENCE_TYPES,
)

random.seed(RANDOM_SEED)

os.makedirs(STEP4_INPUT_DIR, exist_ok=True)
os.makedirs(STEP4_OUTPUT_DIR, exist_ok=True)

# ============================================================
# Output verbosity control
# ============================================================

PRINT_FULL_CONFIG = False
PRINT_RAW_SAMPLE_SUMMARY = False
PRINT_SCENE_EXPORT_SUMMARY = True
SHOW_PREVIEW_HEAD = False
PRINT_SANITY_EXAMPLES = True
MAX_EXAMPLE_CHARS = 2500

# ============================================================
# Concise config summary
# ============================================================

print("Step 4 config loaded.")
print("RANDOM_SEED:", RANDOM_SEED)
print("RUN_MODE:", RUN_MODE)
print("STEP4_INPUT_DIR:", STEP4_INPUT_DIR)
print("STEP4_OUTPUT_DIR:", STEP4_OUTPUT_DIR)
print("SCENES from config:", SCENES)
print("MAIN_PROBE_EVIDENCE_TYPES:", MAIN_PROBE_EVIDENCE_TYPES)

print("\nEvidence type constants:")
print("EXPLICIT_DIRECT:", EXPLICIT_DIRECT)
print("EXPLICIT_INVERSE:", EXPLICIT_INVERSE)
print("IMPLICIT_LABELED:", IMPLICIT_LABELED)
print("IMPLICIT_NONE:", IMPLICIT_NONE)
print("NONE_RELATION_LABEL:", NONE_RELATION_LABEL)

if PRINT_FULL_CONFIG:
    print("\nFull relation config:")
    print("INVERSE_RELATION_MAP:", INVERSE_RELATION_MAP)
    print("DIRECTIONAL_RELATIONS:", DIRECTIONAL_RELATIONS)
    print("SYMMETRIC_OR_WEAK_RELATIONS:", SYMMETRIC_OR_WEAK_RELATIONS)

Step 4 config loaded.
RANDOM_SEED: 42
RUN_MODE: diverse
STEP4_INPUT_DIR: /content/pilot_3/data/step3A_diverse_text_preserve_text_direction
STEP4_OUTPUT_DIR: /content/pilot_3/data/step4_probe_datasets_preserve_text_direction
SCENES from config: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5', 'FloorPlan6']
MAIN_PROBE_EVIDENCE_TYPES: {'explicit_direct', 'explicit_inverse_or_same_sentence_pair'}

Evidence type constants:
EXPLICIT_DIRECT: explicit_direct
EXPLICIT_INVERSE: explicit_inverse_or_same_sentence_pair
IMPLICIT_LABELED: implicit_labeled
IMPLICIT_NONE: implicit_none
NONE_RELATION_LABEL: none


In [3]:
# ============================================================
# Utility functions
# ============================================================

def stable_hash_text(text: str, n: int = 16) -> str:
    """
    Stable short hash for sample ids.
    """
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:n]


def inverse_relation(label: str) -> str:
    """
    Return inverse label if available.
    """
    return INVERSE_RELATION_MAP.get(label, label)


def find_all_spans(text: str, needle: str) -> List[Dict[str, int]]:
    """
    Return all character spans of needle in text.
    """
    spans = []

    if not needle:
        return spans

    start = 0
    while True:
        idx = text.find(needle, start)
        if idx == -1:
            break

        spans.append({
            "start": idx,
            "end": idx + len(needle),
        })
        start = idx + len(needle)

    return spans


def find_first_span(text: str, needle: str) -> Optional[Dict[str, int]]:
    """
    Return first character span of needle in text.
    """
    if not needle:
        return None

    idx = text.find(needle)
    if idx == -1:
        return None

    return {
        "start": idx,
        "end": idx + len(needle),
    }


def relation_pair_key(
    subject_uid: str,
    object_uid: str,
    relation_label: str,
) -> Tuple[str, str, str]:
    """
    Directed relation key using aliases/UIDs.
    """
    return (subject_uid, object_uid, relation_label)


def unordered_pair_key(uid_a: str, uid_b: str) -> str:
    """
    Stable unordered pair key.
    """
    a, b = sorted([uid_a, uid_b])
    return f"{a}|||{b}"


def get_object_mention_index(paragraph_record: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    """
    Build object_uid -> object mention metadata.
    """
    return {
        obj["object_uid"]: obj
        for obj in paragraph_record.get("object_mentions", [])
        if "object_uid" in obj
    }


def safe_get_object_meta(
    object_index: Dict[str, Dict[str, Any]],
    object_uid: str,
) -> Dict[str, Any]:
    """
    Return object metadata from object_mentions if available.
    """
    obj = object_index.get(object_uid, {})

    return {
        "object_uid": object_uid,
        "object_id": obj.get("object_id"),
        "object_type": obj.get("object_type"),
        "mention_text": obj.get("mention_text", object_uid),
    }


def summarize_records(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Summarize a list of Step 4 samples.

    This function is intentionally compact and used only for diagnostics.
    """
    label_counts = Counter()
    evidence_counts = Counter()
    scene_counts = Counter()
    direction_counts = Counter()
    paragraph_counts = Counter()

    for row in records:
        probe_pair = row.get("probe_pair", {})
        evidence = row.get("evidence", {})

        label_counts[probe_pair.get("probe_relation_label", "__missing__")] += 1
        evidence_counts[evidence.get("evidence_type", "__missing__")] += 1
        scene_counts[row.get("scene", "__missing__")] += 1
        direction_counts[evidence.get("probe_direction_relative_to_text", "__missing__")] += 1
        paragraph_counts[row.get("paragraph_id", "__missing__")] += 1

    return {
        "num_samples": len(records),
        "num_paragraphs": len(paragraph_counts),
        "label_counts": dict(label_counts),
        "evidence_counts": dict(evidence_counts),
        "scene_counts": dict(scene_counts),
        "probe_direction_counts": dict(direction_counts),
    }


def write_jsonl(path: str, rows: List[Dict[str, Any]]) -> None:
    """
    Write rows to JSONL.
    """
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_json(path: str, data: Dict[str, Any]) -> None:
    """
    Write JSON.
    """
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


print("Utility functions loaded.")

Utility functions loaded.


In [4]:
# ============================================================
# Build evidence index for explicit_relations_in_text
#
# This cell maps:
#   text explicit direct:
#       text_subject -> text_object = text_relation
#
#   explicit inverse probe:
#       text_object -> text_subject = inverse(text_relation)
#
# It also computes sentence-level and paragraph-level char spans.
# ============================================================

def build_explicit_evidence_index(
    paragraph_record: Dict[str, Any]
) -> Dict[str, Dict[Tuple[str, str, str], Dict[str, Any]]]:
    """
    Build direct and inverse evidence indexes for a paragraph.

    Returns:
        {
            "direct": {
                (text_subject_uid, text_object_uid, text_relation): evidence_record
            },
            "inverse": {
                (text_object_uid, text_subject_uid, inverse_relation): evidence_record
            }
        }
    """
    text = paragraph_record.get("text", "")
    explicit_rels = paragraph_record.get("explicit_relations_in_text", [])

    direct_index = {}
    inverse_index = {}

    cursor = 0

    for rel_idx, rel in enumerate(explicit_rels):
        sentence = rel.get("sentence", "")

        sentence_start = text.find(sentence, cursor)
        if sentence_start == -1:
            sentence_start = text.find(sentence)

        if sentence_start == -1:
            sentence_span = None
        else:
            sentence_end = sentence_start + len(sentence)
            sentence_span = {
                "start": sentence_start,
                "end": sentence_end,
            }
            cursor = sentence_end

        text_subject_uid = rel.get("subject_uid")
        text_object_uid = rel.get("object_uid")
        text_relation = rel.get("relation")

        if not text_subject_uid or not text_object_uid or not text_relation:
            paragraph_id = paragraph_record.get("paragraph_id")
            raise ValueError(
                "Explicit relation is missing subject_uid, object_uid, or relation. "
                f"paragraph_id={paragraph_id}, relation_index={rel_idx}, rel={rel}"
            )

        subj_span_in_sentence = find_first_span(sentence, text_subject_uid)
        obj_span_in_sentence = find_first_span(sentence, text_object_uid)

        if subj_span_in_sentence is not None and sentence_start != -1:
            text_subject_span_in_paragraph = {
                "start": sentence_start + subj_span_in_sentence["start"],
                "end": sentence_start + subj_span_in_sentence["end"],
            }
        else:
            text_subject_span_in_paragraph = None

        if obj_span_in_sentence is not None and sentence_start != -1:
            text_object_span_in_paragraph = {
                "start": sentence_start + obj_span_in_sentence["start"],
                "end": sentence_start + obj_span_in_sentence["end"],
            }
        else:
            text_object_span_in_paragraph = None

        # Surface order can differ from subject-object direction.
        if subj_span_in_sentence is None or obj_span_in_sentence is None:
            surface_order = "unknown"
        elif subj_span_in_sentence["start"] < obj_span_in_sentence["start"]:
            surface_order = "text_subject_before_text_object"
        else:
            surface_order = "text_object_before_text_subject"

        evidence_record = {
            "sentence_index_in_paragraph": rel_idx,
            "evidence_sentence": sentence,
            "evidence_sentence_char_span_in_paragraph": sentence_span,

            "template": rel.get("template"),
            "template_index": rel.get("template_index"),
            "surface_order": surface_order,

            "text_subject_uid": text_subject_uid,
            "text_subject_id": rel.get("subject_id"),
            "text_subject_type": rel.get("subject_type"),
            "text_subject_span_in_evidence_sentence": subj_span_in_sentence,
            "text_subject_span_in_paragraph": text_subject_span_in_paragraph,

            "text_object_uid": text_object_uid,
            "text_object_id": rel.get("object_id"),
            "text_object_type": rel.get("object_type"),
            "text_object_span_in_evidence_sentence": obj_span_in_sentence,
            "text_object_span_in_paragraph": text_object_span_in_paragraph,

            "text_relation_label": text_relation,
            "was_swapped_for_text": rel.get("was_swapped_for_text"),
            "source_relation": rel.get("source_relation"),
        }

        direct_key = relation_pair_key(
            text_subject_uid,
            text_object_uid,
            text_relation,
        )

        inv_key = relation_pair_key(
            text_object_uid,
            text_subject_uid,
            inverse_relation(text_relation),
        )

        direct_index[direct_key] = evidence_record
        inverse_index[inv_key] = evidence_record

    return {
        "direct": direct_index,
        "inverse": inverse_index,
    }


print("Explicit evidence index builder loaded.")

Explicit evidence index builder loaded.


In [5]:
# ============================================================
# Build one Step 4 probe sample from one Step 3 pair_target
# ============================================================

def infer_probe_direction_relative_to_text(
    pair_evidence_type: str,
) -> str:
    """
    Normalize evidence type into probe direction.
    """
    if pair_evidence_type == EXPLICIT_DIRECT:
        return "direct"
    if pair_evidence_type == EXPLICIT_INVERSE:
        return "inverse"
    if pair_evidence_type == IMPLICIT_LABELED:
        return "implicit_labeled"
    if pair_evidence_type == IMPLICIT_NONE:
        return "implicit_none"
    return "unknown"


def build_step4_sample(
    paragraph_record: Dict[str, Any],
    pair_target: Dict[str, Any],
    evidence_index: Dict[str, Dict[Tuple[str, str, str], Dict[str, Any]]],
) -> Dict[str, Any]:
    """
    Build a Step 4 probe sample.

    The output explicitly separates:
        - text direction
        - probe direction
        - label direction
    """
    paragraph_id = paragraph_record["paragraph_id"]
    text = paragraph_record["text"]
    object_index = get_object_mention_index(paragraph_record)

    pt_subject_uid = pair_target.get("subject_uid")
    pt_object_uid = pair_target.get("object_uid")
    pt_label = pair_target.get("relation_label")
    pair_evidence_type = pair_target.get("pair_evidence_type")

    if not pt_subject_uid or not pt_object_uid or pt_label is None:
        raise ValueError(
            "pair_target is missing subject_uid, object_uid, or relation_label. "
            f"paragraph_id={paragraph_id}, pair_target={pair_target}"
        )

    if pair_evidence_type not in {
        EXPLICIT_DIRECT,
        EXPLICIT_INVERSE,
        IMPLICIT_LABELED,
        IMPLICIT_NONE,
    }:
        raise ValueError(
            "Unknown pair_evidence_type. "
            f"paragraph_id={paragraph_id}, pair_evidence_type={pair_evidence_type}, "
            f"pair_target={pair_target}"
        )

    probe_direction = infer_probe_direction_relative_to_text(pair_evidence_type)

    probe_subject_meta = safe_get_object_meta(object_index, pt_subject_uid)
    probe_object_meta = safe_get_object_meta(object_index, pt_object_uid)

    # Probe mention spans in the full paragraph.
    # For explicit evidence, the evidence-specific span will be added below.
    probe_subject_all_spans = find_all_spans(text, pt_subject_uid)
    probe_object_all_spans = find_all_spans(text, pt_object_uid)

    lookup_key = relation_pair_key(
        pt_subject_uid,
        pt_object_uid,
        pt_label,
    )

    evidence_record = None

    if pair_evidence_type == EXPLICIT_DIRECT:
        evidence_record = evidence_index["direct"].get(lookup_key)

    elif pair_evidence_type == EXPLICIT_INVERSE:
        evidence_record = evidence_index["inverse"].get(lookup_key)

    # Explicit samples must have a matching evidence sentence.
    if pair_evidence_type in {EXPLICIT_DIRECT, EXPLICIT_INVERSE} and evidence_record is None:
        raise KeyError(
            "Explicit pair_target could not be matched to an evidence sentence. "
            f"paragraph_id={paragraph_id}, "
            f"pair_evidence_type={pair_evidence_type}, "
            f"lookup_key={lookup_key}, "
            f"available_direct_keys={list(evidence_index['direct'].keys())[:5]}, "
            f"available_inverse_keys={list(evidence_index['inverse'].keys())[:5]}"
        )

    # Fill explicit evidence fields if available.
    if evidence_record is not None:
        text_subject_uid = evidence_record["text_subject_uid"]
        text_object_uid = evidence_record["text_object_uid"]
        text_relation_label = evidence_record["text_relation_label"]

        pair_group_id = (
            f"{paragraph_id}|"
            f"{text_subject_uid}|"
            f"{text_object_uid}|"
            f"{text_relation_label}"
        )

        if probe_direction == "direct":
            probe_subject_span_in_evidence_sentence = evidence_record.get(
                "text_subject_span_in_evidence_sentence"
            )
            probe_object_span_in_evidence_sentence = evidence_record.get(
                "text_object_span_in_evidence_sentence"
            )
            probe_subject_span_in_paragraph = evidence_record.get(
                "text_subject_span_in_paragraph"
            )
            probe_object_span_in_paragraph = evidence_record.get(
                "text_object_span_in_paragraph"
            )

        elif probe_direction == "inverse":
            probe_subject_span_in_evidence_sentence = evidence_record.get(
                "text_object_span_in_evidence_sentence"
            )
            probe_object_span_in_evidence_sentence = evidence_record.get(
                "text_subject_span_in_evidence_sentence"
            )
            probe_subject_span_in_paragraph = evidence_record.get(
                "text_object_span_in_paragraph"
            )
            probe_object_span_in_paragraph = evidence_record.get(
                "text_subject_span_in_paragraph"
            )

        else:
            raise ValueError(
                "Explicit evidence has invalid probe_direction. "
                f"paragraph_id={paragraph_id}, probe_direction={probe_direction}"
            )

        evidence_block = {
            "evidence_type": pair_evidence_type,
            "probe_direction_relative_to_text": probe_direction,

            "has_explicit_evidence_sentence": True,
            "evidence_sentence": evidence_record.get("evidence_sentence"),
            "sentence_index_in_paragraph": evidence_record.get("sentence_index_in_paragraph"),
            "evidence_sentence_char_span_in_paragraph": evidence_record.get(
                "evidence_sentence_char_span_in_paragraph"
            ),

            "template": evidence_record.get("template"),
            "template_index": evidence_record.get("template_index"),
            "surface_order": evidence_record.get("surface_order"),

            "text_subject_uid": evidence_record.get("text_subject_uid"),
            "text_subject_id": evidence_record.get("text_subject_id"),
            "text_subject_type": evidence_record.get("text_subject_type"),
            "text_subject_span_in_evidence_sentence": evidence_record.get(
                "text_subject_span_in_evidence_sentence"
            ),
            "text_subject_span_in_paragraph": evidence_record.get(
                "text_subject_span_in_paragraph"
            ),

            "text_object_uid": evidence_record.get("text_object_uid"),
            "text_object_id": evidence_record.get("text_object_id"),
            "text_object_type": evidence_record.get("text_object_type"),
            "text_object_span_in_evidence_sentence": evidence_record.get(
                "text_object_span_in_evidence_sentence"
            ),
            "text_object_span_in_paragraph": evidence_record.get(
                "text_object_span_in_paragraph"
            ),

            "text_relation_label": evidence_record.get("text_relation_label"),
            "was_swapped_for_text": evidence_record.get("was_swapped_for_text"),
        }

    else:
        # Implicit samples do not have a direct evidence sentence.
        pair_group_id = (
            f"{paragraph_id}|"
            f"{unordered_pair_key(pt_subject_uid, pt_object_uid)}|"
            f"{pt_label}|"
            f"{pair_evidence_type}"
        )

        evidence_block = {
            "evidence_type": pair_evidence_type,
            "probe_direction_relative_to_text": probe_direction,

            "has_explicit_evidence_sentence": False,
            "evidence_sentence": None,
            "sentence_index_in_paragraph": None,
            "evidence_sentence_char_span_in_paragraph": None,

            "template": None,
            "template_index": None,
            "surface_order": None,

            "text_subject_uid": None,
            "text_subject_id": None,
            "text_subject_type": None,
            "text_subject_span_in_evidence_sentence": None,
            "text_subject_span_in_paragraph": None,

            "text_object_uid": None,
            "text_object_id": None,
            "text_object_type": None,
            "text_object_span_in_evidence_sentence": None,
            "text_object_span_in_paragraph": None,

            "text_relation_label": None,
            "was_swapped_for_text": None,
        }

        probe_subject_span_in_evidence_sentence = None
        probe_object_span_in_evidence_sentence = None
        probe_subject_span_in_paragraph = None
        probe_object_span_in_paragraph = None

    is_symmetric_relation = pt_label in SYMMETRIC_OR_WEAK_RELATIONS
    is_directional_relation = pt_label in DIRECTIONAL_RELATIONS

    sample_core = (
        f"{paragraph_id}|"
        f"{pt_subject_uid}|"
        f"{pt_object_uid}|"
        f"{pt_label}|"
        f"{pair_evidence_type}|"
        f"{probe_direction}"
    )

    sample_id = f"step4_{stable_hash_text(sample_core, n=20)}"

    source_relation_summary = pair_target.get("source_relation_summary")

    sample = {
        "sample_id": sample_id,
        "pair_group_id": pair_group_id,

        "scene": paragraph_record.get("scene"),
        "experiment": paragraph_record.get("experiment"),
        "generation_mode": paragraph_record.get("generation_mode"),

        "paragraph_id": paragraph_id,
        "chunk_id": paragraph_record.get("chunk_id"),
        "cluster_id": paragraph_record.get("cluster_id"),
        "grid_i": paragraph_record.get("grid_i"),
        "grid_j": paragraph_record.get("grid_j"),
        "paragraph_index_within_cluster": paragraph_record.get(
            "paragraph_index_within_cluster"
        ),

        # Full LLM input text.
        "text": text,

        # Evidence sentence and text-direction metadata.
        "evidence": evidence_block,

        # Probe direction metadata.
        "probe_pair": {
            "probe_subject_uid": pt_subject_uid,
            "probe_subject_id": pair_target.get("subject_id"),
            "probe_subject_type": pair_target.get("subject_type"),
            "probe_subject_mention_text": probe_subject_meta.get("mention_text"),
            "probe_subject_all_char_spans_in_paragraph": probe_subject_all_spans,
            "probe_subject_span_in_evidence_sentence": probe_subject_span_in_evidence_sentence,
            "probe_subject_span_in_paragraph": probe_subject_span_in_paragraph,

            "probe_object_uid": pt_object_uid,
            "probe_object_id": pair_target.get("object_id"),
            "probe_object_type": pair_target.get("object_type"),
            "probe_object_mention_text": probe_object_meta.get("mention_text"),
            "probe_object_all_char_spans_in_paragraph": probe_object_all_spans,
            "probe_object_span_in_evidence_sentence": probe_object_span_in_evidence_sentence,
            "probe_object_span_in_paragraph": probe_object_span_in_paragraph,

            "probe_relation_label": pt_label,
            "label_space": "directed_spatial_relation",

            "has_relation_label": pair_target.get("has_relation_label"),
            "is_inverse_of_text_relation": probe_direction == "inverse",
            "is_symmetric_relation": is_symmetric_relation,
            "is_directional_relation": is_directional_relation,
        },

        # Source relation / geometry from Step 2/Step 3.
        "source_relation": {
            "relation_source": pair_target.get("relation_source"),
            "source_relation_summary": source_relation_summary,
        },

        "geometry": {
            "has_geometry_label": pair_target.get("geometry_label") is not None,
            "geometry_label": pair_target.get("geometry_label"),
        },
    }

    return sample


print("Step 4 sample builder loaded.")

Step 4 sample builder loaded.


In [6]:
# ============================================================
# Load all existing Step 3 files and build Step 4 samples
#
# This cell scans STEP4_INPUT_DIR and processes all files matching:
#   *_experiment_A_text_{RUN_MODE}.json
#
# It does NOT rely on config.SCENES.
# It prints a concise summary of discovered and processed files.
# ============================================================

import glob
import os
import json
from typing import List, Dict, Any


def discover_step3_paths() -> List[str]:
    pattern = os.path.join(
        STEP4_INPUT_DIR,
        f"*_experiment_A_text_{RUN_MODE}.json"
    )
    return sorted(glob.glob(pattern))


def validate_step3_paragraph_schema(paragraph: Dict[str, Any]) -> None:
    """
    Validate the minimum Step 3 paragraph schema required by Step 4.

    This catches upstream Step 3 schema changes early, before Step 4 silently
    creates incorrect samples.
    """
    required_paragraph_keys = {
        "paragraph_id",
        "scene",
        "chunk_id",
        "cluster_id",
        "text",
        "object_mentions",
        "explicit_relations_in_text",
        "pair_targets",
    }

    missing_paragraph_keys = [
        key for key in required_paragraph_keys
        if key not in paragraph
    ]

    if missing_paragraph_keys:
        raise KeyError(
            f"Step 3 paragraph is missing required keys: {missing_paragraph_keys}. "
            f"paragraph_id={paragraph.get('paragraph_id')}"
        )

    for idx, rel in enumerate(paragraph.get("explicit_relations_in_text", [])):
        required_relation_keys = {
            "sentence",
            "template",
            "template_index",
            "subject_uid",
            "subject_id",
            "subject_type",
            "relation",
            "object_uid",
            "object_id",
            "object_type",
            "was_swapped_for_text",
            "source_relation",
        }

        missing_relation_keys = [
            key for key in required_relation_keys
            if key not in rel
        ]

        if missing_relation_keys:
            raise KeyError(
                f"Step 3 explicit relation is missing required keys: "
                f"{missing_relation_keys}. "
                f"paragraph_id={paragraph.get('paragraph_id')}, relation_index={idx}"
            )

    for idx, pair_target in enumerate(paragraph.get("pair_targets", [])):
        required_pair_keys = {
            "subject_uid",
            "subject_id",
            "subject_type",
            "object_uid",
            "object_id",
            "object_type",
            "relation_label",
            "has_relation_label",
            "relation_source",
            "pair_evidence_type",
            "geometry_label",
            "source_relation_summary",
        }

        missing_pair_keys = [
            key for key in required_pair_keys
            if key not in pair_target
        ]

        if missing_pair_keys:
            raise KeyError(
                f"Step 3 pair_target is missing required keys: "
                f"{missing_pair_keys}. "
                f"paragraph_id={paragraph.get('paragraph_id')}, pair_index={idx}"
            )


step3_paths = discover_step3_paths()

if not step3_paths:
    raise FileNotFoundError(
        "No Step 3 files found.\n"
        f"STEP4_INPUT_DIR={STEP4_INPUT_DIR}\n"
        f"Expected pattern=*_experiment_A_text_{RUN_MODE}.json"
    )

print("Discovered Step 3 files")
print("STEP4_INPUT_DIR:", STEP4_INPUT_DIR)
print("RUN_MODE:", RUN_MODE)
print("num_files:", len(step3_paths))

for path in step3_paths:
    print("-", os.path.basename(path))


all_step4_samples = []
processed_file_summaries = []

for step3_path in step3_paths:
    with open(step3_path, "r", encoding="utf-8") as f:
        step3_data = json.load(f)

    scene = step3_data["scene"]
    paragraphs = step3_data.get("paragraphs", [])

    scene_sample_count = 0
    scene_explicit_relation_count = 0
    scene_pair_target_count = 0

    for paragraph in paragraphs:
        validate_step3_paragraph_schema(paragraph)

        pair_targets = paragraph.get("pair_targets", [])
        explicit_relations = paragraph.get("explicit_relations_in_text", [])

        scene_pair_target_count += len(pair_targets)
        scene_explicit_relation_count += len(explicit_relations)

        # Build paragraph-level evidence index once.
        evidence_index = build_explicit_evidence_index(paragraph)

        for pair_target in pair_targets:
            sample = build_step4_sample(
                paragraph_record=paragraph,
                pair_target=pair_target,
                evidence_index=evidence_index,
            )

            sample["source_step3_file"] = os.path.basename(step3_path)
            sample["source_step3_scene"] = scene

            all_step4_samples.append(sample)
            scene_sample_count += 1

    file_summary = {
        "source_step3_file": os.path.basename(step3_path),
        "scene": scene,
        "num_paragraphs": len(paragraphs),
        "num_explicit_relations_in_text": scene_explicit_relation_count,
        "num_pair_targets": scene_pair_target_count,
        "num_raw_step4_samples": scene_sample_count,
    }

    processed_file_summaries.append(file_summary)


print("\nProcessed file summary:")
for item in processed_file_summaries:
    print(
        f"- {item['source_step3_file']} | "
        f"scene={item['scene']} | "
        f"paragraphs={item['num_paragraphs']} | "
        f"explicit_relations={item['num_explicit_relations_in_text']} | "
        f"pair_targets={item['num_pair_targets']} | "
        f"raw_samples={item['num_raw_step4_samples']}"
    )

print("\nTotal processed Step 3 files:", len(processed_file_summaries))
print("Total raw Step 4 samples before filtering:", len(all_step4_samples))

if PRINT_RAW_SAMPLE_SUMMARY:
    print("\nRaw sample summary:")
    print(json.dumps(summarize_records(all_step4_samples), indent=2, ensure_ascii=False))

Discovered Step 3 files
STEP4_INPUT_DIR: /content/pilot_3/data/step3A_diverse_text_preserve_text_direction
RUN_MODE: diverse
num_files: 6
- FloorPlan1_experiment_A_text_diverse.json
- FloorPlan2_experiment_A_text_diverse.json
- FloorPlan3_experiment_A_text_diverse.json
- FloorPlan4_experiment_A_text_diverse.json
- FloorPlan5_experiment_A_text_diverse.json
- FloorPlan6_experiment_A_text_diverse.json

Processed file summary:
- FloorPlan1_experiment_A_text_diverse.json | scene=FloorPlan1 | paragraphs=24 | explicit_relations=354 | pair_targets=2666 | raw_samples=2666
- FloorPlan2_experiment_A_text_diverse.json | scene=FloorPlan2 | paragraphs=20 | explicit_relations=292 | pair_targets=2074 | raw_samples=2074
- FloorPlan3_experiment_A_text_diverse.json | scene=FloorPlan3 | paragraphs=24 | explicit_relations=293 | pair_targets=1410 | raw_samples=1410
- FloorPlan4_experiment_A_text_diverse.json | scene=FloorPlan4 | paragraphs=20 | explicit_relations=293 | pair_targets=2282 | raw_samples=2282
-

In [7]:
# ============================================================
# Export one Step 4 JSONL + one human-readable JSON per FloorPlan
#
# Main probe export is controlled by config.MAIN_PROBE_EVIDENCE_TYPES.
#
# Current intended setting:
#   explicit_direct
#   explicit_inverse_or_same_sentence_pair
#
# It drops:
#   implicit_labeled
#   implicit_none
# ============================================================

def group_samples_by_scene(samples: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    grouped = defaultdict(list)

    for row in samples:
        grouped[row["scene"]].append(row)

    return dict(grouped)


def keep_main_probe_sample(row: Dict[str, Any]) -> bool:
    """
    Keep samples according to config.MAIN_PROBE_EVIDENCE_TYPES.
    Also drop label == none.
    """
    evidence_type = row["evidence"]["evidence_type"]
    label = row["probe_pair"]["probe_relation_label"]

    if evidence_type not in MAIN_PROBE_EVIDENCE_TYPES:
        return False

    if label == NONE_RELATION_LABEL:
        return False

    return True


main_probe_samples = [
    row for row in all_step4_samples
    if keep_main_probe_sample(row)
]

scene_groups = group_samples_by_scene(main_probe_samples)

export_summary = {
    "step4_config": {
        "random_seed": RANDOM_SEED,
        "scenes_from_config": SCENES,
        "run_mode": RUN_MODE,

        "step4_input_dir": STEP4_INPUT_DIR,
        "step4_output_dir": STEP4_OUTPUT_DIR,

        "none_relation_label": NONE_RELATION_LABEL,
        "inverse_relation_map": INVERSE_RELATION_MAP,

        "export_mode": "main_probe_from_config",
        "description": (
            "Step 4 constructs probe samples from Step 3 pair_targets. "
            "One JSONL and one human-readable JSON file are exported per FloorPlan. "
            "The kept evidence types are controlled by config.MAIN_PROBE_EVIDENCE_TYPES. "
            "Samples with relation label == none are excluded."
        ),

        "kept_evidence_types": sorted(MAIN_PROBE_EVIDENCE_TYPES),
        "dropped_evidence_types": sorted(
            {
                EXPLICIT_DIRECT,
                EXPLICIT_INVERSE,
                IMPLICIT_LABELED,
                IMPLICIT_NONE,
            } - set(MAIN_PROBE_EVIDENCE_TYPES)
        ),
    },

    # Actual Step 3 files discovered and processed.
    "source_files": [os.path.basename(p) for p in step3_paths],
    "processed_step3_files": processed_file_summaries,

    # Full summaries are saved here, but not repeatedly printed.
    "overall_before_filtering": summarize_records(all_step4_samples),
    "overall_after_filtering": summarize_records(main_probe_samples),

    "scenes": {},
}

for scene, rows in sorted(scene_groups.items()):
    jsonl_path = os.path.join(
        STEP4_OUTPUT_DIR,
        f"{scene}_step4_probe_samples.jsonl"
    )

    json_path = os.path.join(
        STEP4_OUTPUT_DIR,
        f"{scene}_step4_probe_samples.json"
    )

    # Machine-friendly JSONL.
    write_jsonl(jsonl_path, rows)

    # Human-readable JSON.
    write_json(json_path, rows)

    summary = summarize_records(rows)

    export_summary["scenes"][scene] = {
        "jsonl_path": jsonl_path,
        "json_path": json_path,
        **summary,
    }

    if PRINT_SCENE_EXPORT_SUMMARY:
        print(
            f"Exported {scene}: "
            f"samples={len(rows)} | "
            f"jsonl={os.path.basename(jsonl_path)} | "
            f"json={os.path.basename(json_path)}"
        )

summary_path = os.path.join(STEP4_OUTPUT_DIR, "step4_export_summary.json")
write_json(summary_path, export_summary)

print("\nExport summary written to:")
print(summary_path)

print("\nOverall export summary:")
print("Raw samples before filtering:", len(all_step4_samples))
print("Main probe samples after filtering:", len(main_probe_samples))
print("Scenes exported:", sorted(scene_groups.keys()))
print("Number of scenes exported:", len(scene_groups))

print("\nMain probe summary after filtering:")
print(json.dumps(summarize_records(main_probe_samples), indent=2, ensure_ascii=False))

Exported FloorPlan1: samples=708 | jsonl=FloorPlan1_step4_probe_samples.jsonl | json=FloorPlan1_step4_probe_samples.json
Exported FloorPlan2: samples=582 | jsonl=FloorPlan2_step4_probe_samples.jsonl | json=FloorPlan2_step4_probe_samples.json
Exported FloorPlan3: samples=564 | jsonl=FloorPlan3_step4_probe_samples.jsonl | json=FloorPlan3_step4_probe_samples.json
Exported FloorPlan4: samples=580 | jsonl=FloorPlan4_step4_probe_samples.jsonl | json=FloorPlan4_step4_probe_samples.json
Exported FloorPlan5: samples=708 | jsonl=FloorPlan5_step4_probe_samples.jsonl | json=FloorPlan5_step4_probe_samples.json
Exported FloorPlan6: samples=640 | jsonl=FloorPlan6_step4_probe_samples.jsonl | json=FloorPlan6_step4_probe_samples.json

Export summary written to:
/content/pilot_3/data/step4_probe_datasets_preserve_text_direction/step4_export_summary.json

Overall export summary:
Raw samples before filtering: 13208
Main probe samples after filtering: 3782
Scenes exported: ['FloorPlan1', 'FloorPlan2', 'Floo

In [8]:
# ============================================================
# Export compact CSV previews per FloorPlan
#
# These CSV files are for human inspection only.
# The JSONL / JSON files remain the actual probe datasets.
#
# Preview is based on main_probe_samples.
# ============================================================

import pandas as pd

def make_preview_row(row: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "sample_id": row["sample_id"],
        "scene": row["scene"],
        "paragraph_id": row["paragraph_id"],
        "chunk_id": row["chunk_id"],
        "cluster_id": row["cluster_id"],

        "evidence_type": row["evidence"]["evidence_type"],
        "probe_direction": row["evidence"]["probe_direction_relative_to_text"],

        "probe_relation_label": row["probe_pair"]["probe_relation_label"],
        "probe_subject_uid": row["probe_pair"]["probe_subject_uid"],
        "probe_object_uid": row["probe_pair"]["probe_object_uid"],

        "text_relation_label": row["evidence"]["text_relation_label"],
        "text_subject_uid": row["evidence"]["text_subject_uid"],
        "text_object_uid": row["evidence"]["text_object_uid"],

        "has_explicit_evidence_sentence": row["evidence"]["has_explicit_evidence_sentence"],
        "evidence_sentence": row["evidence"]["evidence_sentence"],

        "template_index": row["evidence"]["template_index"],
        "surface_order": row["evidence"]["surface_order"],

        "was_swapped_for_text": row["evidence"]["was_swapped_for_text"],

        "has_geometry_label": row["geometry"]["has_geometry_label"],

        "text": row["text"],
    }


preview_csv_summaries = []

for scene, rows in sorted(scene_groups.items()):
    preview_rows = [
        make_preview_row(row)
        for row in rows
    ]

    df = pd.DataFrame(preview_rows)

    csv_path = os.path.join(
        STEP4_OUTPUT_DIR,
        f"{scene}_step4_probe_samples_preview.csv"
    )

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    preview_csv_summaries.append({
        "scene": scene,
        "csv_path": csv_path,
        "num_rows": len(df),
    })

    print(
        f"Preview CSV exported: {scene} | "
        f"rows={len(df)} | "
        f"file={os.path.basename(csv_path)}"
    )

    if SHOW_PREVIEW_HEAD:
        display(df.head(10))


print("\nPreview CSV summary:")
for item in preview_csv_summaries:
    print(
        f"- {item['scene']} | "
        f"rows={item['num_rows']} | "
        f"path={item['csv_path']}"
    )

Preview CSV exported: FloorPlan1 | rows=708 | file=FloorPlan1_step4_probe_samples_preview.csv
Preview CSV exported: FloorPlan2 | rows=582 | file=FloorPlan2_step4_probe_samples_preview.csv
Preview CSV exported: FloorPlan3 | rows=564 | file=FloorPlan3_step4_probe_samples_preview.csv
Preview CSV exported: FloorPlan4 | rows=580 | file=FloorPlan4_step4_probe_samples_preview.csv
Preview CSV exported: FloorPlan5 | rows=708 | file=FloorPlan5_step4_probe_samples_preview.csv
Preview CSV exported: FloorPlan6 | rows=640 | file=FloorPlan6_step4_probe_samples_preview.csv

Preview CSV summary:
- FloorPlan1 | rows=708 | path=/content/pilot_3/data/step4_probe_datasets_preserve_text_direction/FloorPlan1_step4_probe_samples_preview.csv
- FloorPlan2 | rows=582 | path=/content/pilot_3/data/step4_probe_datasets_preserve_text_direction/FloorPlan2_step4_probe_samples_preview.csv
- FloorPlan3 | rows=564 | path=/content/pilot_3/data/step4_probe_datasets_preserve_text_direction/FloorPlan3_step4_probe_samples_pre

In [9]:
# ============================================================
# Sanity checks for main probe export
#
# This cell avoids repeating export summaries.
# It focuses on evidence type distribution, label distribution,
# direct/inverse pairing, and a small number of global examples.
# ============================================================

def print_example_global(
    samples: List[Dict[str, Any]],
    evidence_type: str,
    max_chars: int = 2500,
) -> None:
    """
    Print one global example for a given evidence type.
    """
    for row in samples:
        if row["evidence"]["evidence_type"] == evidence_type:
            print(f"\nGlobal example for evidence_type={evidence_type}")
            print(json.dumps(row, indent=2, ensure_ascii=False)[:max_chars])
            return

    print(f"\nNo global example found for evidence_type={evidence_type}")


def check_explicit_pair_groups(samples: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Check whether explicit direct/inverse samples are paired by pair_group_id.
    """
    explicit_rows = [
        row for row in samples
        if row["evidence"]["evidence_type"] in {EXPLICIT_DIRECT, EXPLICIT_INVERSE}
    ]

    group_counts = defaultdict(Counter)

    for row in explicit_rows:
        group_id = row["pair_group_id"]
        evidence_type = row["evidence"]["evidence_type"]
        group_counts[group_id][evidence_type] += 1

    num_groups = len(group_counts)
    complete_groups = 0
    incomplete_groups = []

    for group_id, counts in group_counts.items():
        if counts[EXPLICIT_DIRECT] >= 1 and counts[EXPLICIT_INVERSE] >= 1:
            complete_groups += 1
        else:
            incomplete_groups.append({
                "pair_group_id": group_id,
                "counts": dict(counts),
            })

    return {
        "num_explicit_pair_groups": num_groups,
        "num_complete_direct_inverse_groups": complete_groups,
        "num_incomplete_groups": len(incomplete_groups),
        "first_incomplete_groups": incomplete_groups[:10],
    }


print("=" * 100)
print("Sanity check summary")
print("Main probe samples:", len(main_probe_samples))
print("Scenes:", sorted(scene_groups.keys()))

print("\nEvidence type counts:")
print(json.dumps(
    dict(Counter(row["evidence"]["evidence_type"] for row in main_probe_samples)),
    indent=2,
    ensure_ascii=False
))

print("\nProbe label counts:")
print(json.dumps(
    dict(Counter(row["probe_pair"]["probe_relation_label"] for row in main_probe_samples)),
    indent=2,
    ensure_ascii=False
))

print("\nSamples per scene:")
for scene, rows in sorted(scene_groups.items()):
    print(f"- {scene}: {len(rows)} samples")

if MAIN_PROBE_EVIDENCE_TYPES == {EXPLICIT_DIRECT, EXPLICIT_INVERSE}:
    print("\nDirect/inverse group check, overall:")
    print(json.dumps(
        check_explicit_pair_groups(main_probe_samples),
        indent=2,
        ensure_ascii=False
    ))

    print("\nDirect/inverse group check, per scene:")
    for scene, rows in sorted(scene_groups.items()):
        group_check = check_explicit_pair_groups(rows)
        print(
            f"- {scene}: "
            f"groups={group_check['num_explicit_pair_groups']}, "
            f"complete={group_check['num_complete_direct_inverse_groups']}, "
            f"incomplete={group_check['num_incomplete_groups']}"
        )

if PRINT_SANITY_EXAMPLES:
    for evidence_type in sorted(MAIN_PROBE_EVIDENCE_TYPES):
        print_example_global(
            main_probe_samples,
            evidence_type,
            max_chars=MAX_EXAMPLE_CHARS,
        )

Sanity check summary
Main probe samples: 3782
Scenes: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5', 'FloorPlan6']

Evidence type counts:
{
  "explicit_inverse_or_same_sentence_pair": 1872,
  "explicit_direct": 1910
}

Probe label counts:
{
  "above": 330,
  "right_of": 951,
  "contains": 48,
  "on": 435,
  "supports": 435,
  "below": 330,
  "in": 48,
  "left_of": 951,
  "near": 254
}

Samples per scene:
- FloorPlan1: 708 samples
- FloorPlan2: 582 samples
- FloorPlan3: 564 samples
- FloorPlan4: 580 samples
- FloorPlan5: 708 samples
- FloorPlan6: 640 samples

Direct/inverse group check, overall:
{
  "num_explicit_pair_groups": 1910,
  "num_complete_direct_inverse_groups": 1872,
  "num_incomplete_groups": 38,
  "first_incomplete_groups": [
    {
      "pair_group_id": "FloorPlan2_FloorPlan2_chunk_1_1_cluster_0_A_diverse_2|bread_1|egg_1|near",
      "counts": {
        "explicit_direct": 1
      }
    },
    {
      "pair_group_id": "FloorPlan2_FloorPlan2_chunk_1_1

In [11]:
# ============================================================
# Zip Step 4 outputs for download
# ============================================================

from google.colab import files

output_files = sorted(os.listdir(STEP4_OUTPUT_DIR))

if not output_files:
    raise FileNotFoundError(
        f"No files found in STEP4_OUTPUT_DIR: {STEP4_OUTPUT_DIR}"
    )

zip_base = "/content/p4_settingA_step4"

zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=STEP4_OUTPUT_DIR,
)

print("Created zip:", zip_path)
print("STEP4_OUTPUT_DIR:", STEP4_OUTPUT_DIR)
print("Number of files included:", len(output_files))

print("\nFiles included:")
for name in output_files:
    print("-", name)

files.download(zip_path)

Created zip: /content/p4_settingA_step4.zip
STEP4_OUTPUT_DIR: /content/pilot_3/data/step4_probe_datasets_preserve_text_direction
Number of files included: 19

Files included:
- FloorPlan1_step4_probe_samples.json
- FloorPlan1_step4_probe_samples.jsonl
- FloorPlan1_step4_probe_samples_preview.csv
- FloorPlan2_step4_probe_samples.json
- FloorPlan2_step4_probe_samples.jsonl
- FloorPlan2_step4_probe_samples_preview.csv
- FloorPlan3_step4_probe_samples.json
- FloorPlan3_step4_probe_samples.jsonl
- FloorPlan3_step4_probe_samples_preview.csv
- FloorPlan4_step4_probe_samples.json
- FloorPlan4_step4_probe_samples.jsonl
- FloorPlan4_step4_probe_samples_preview.csv
- FloorPlan5_step4_probe_samples.json
- FloorPlan5_step4_probe_samples.jsonl
- FloorPlan5_step4_probe_samples_preview.csv
- FloorPlan6_step4_probe_samples.json
- FloorPlan6_step4_probe_samples.jsonl
- FloorPlan6_step4_probe_samples_preview.csv
- step4_export_summary.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>